In [ ]:
import matplotlib.pyplot as pyplot
import pandas as pd
import numpy as np
import requests
from google.colab import drive

In [ ]:
drive.mount('/content/drive/')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# request % without a vehicle data from the census
KEY = "get individual census key"
url = "https://api.census.gov/data/2018/acs/acs5"

# B25044_003E: TENURE BY VEHICLES AVAILABLE (owner-occupied) (no vehicles)
# B25044_010E: TENURE BY VEHICLES AVAILABLE (renter-occupied) (no vehicles)
# B25044_001E: TENURE BY VEHICLES AVAILABLE (overall total)
params = {
    "get": "NAME,B25044_001E,B25044_003E,B25044_010E",
    "for": "county:*",
    "key": KEY
}

r = requests.get(url, params=params)
rows = r.json() # first row is headers, rest is data
vehicle_df = pd.DataFrame(rows[1:], columns=rows[0])

# combine state and county to get GEOID
# str.zfill(2) ensures that every state is 2 characters; if it comes as 1, becomes 01
# same for county, ensuring both combine to 5 characters
vehicle_df["GEOID"] = vehicle_df["state"].str.zfill(2) + vehicle_df["county"].str.zfill(3)

# all values come as strings; numerical values should be converted to numbers
for c in ["B25044_001E", "B25044_003E", "B25044_010E"]:
  vehicle_df[c] = pd.to_numeric(vehicle_df[c])

# basically (rented households w/ no car + owned households w/ no car) / (total households)
vehicle_df["pct_no_vehicle"] = (vehicle_df["B25044_003E"] + vehicle_df["B25044_010E"]) / vehicle_df["B25044_001E"]

vehicle_df[vehicle_df["GEOID"] == "48317"]

In [ ]:
vehicle_df.sort_values(by="pct_no_vehicle")

In [ ]:
# make that the counties with 0% percent are legit; all are small car-dependent rural counties so seems valid
vehicle_df[vehicle_df["pct_no_vehicle"] == 0]

In [ ]:
vehicle_df = vehicle_df[["GEOID", "pct_no_vehicle"]]

In [ ]:
vehicle_df.to_parquet("/content/drive/MyDrive/Nino/cleaned_data/vehicle_static.parquet")

In [ ]:
# merge with the other static features: HAND and distance to coast
hand_coast = pd.read_parquet("/content/drive/MyDrive/Nino/cleaned_data/hand_coast_static.parquet")
static_all = hand_coast.merge(vehicle_df, on="GEOID", how="left")
static_all

In [ ]:
# make sure there was no fips mismatch
print(static_all["pct_no_vehicle"].isna().sum())

In [ ]:
static_all.to_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/static_all.parquet")